# 1. Introduction

Ce notebook a pour objectif d'entraîner un premier modèle d'intelligence artificielle afin de prédire un signal de marché à partir des indicateurs techniques créés précédemment.

Le modèle utilisé pour ce prototype est Random Forest.

In [1]:
#Importer les bibliothèques

In [5]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 3. Chargement des données enrichies

Chargement des fichiers contenant les indicateurs techniques créés dans le notebook 04.

In [9]:
PROJECT_ROOT = Path.cwd().parent

FEATURES_DIR = PROJECT_ROOT / "data" / "features"

gold = pd.read_csv(FEATURES_DIR / "gold_features.csv")
silver = pd.read_csv(FEATURES_DIR / "silver_features.csv")

print("Gold :", gold.shape)
print("Silver :", silver.shape)

display(gold.head())

Gold : (11423, 13)
Silver : (11420, 13)


,Datetime,Close,High,Low,Open,Volume,Ticker,EMA20,EMA50,RSI,ATR,Target,EMA200
0,2024-05-30 04:00:00+00:00,2333.500000,2335.100098,2332.600098,2332.899902,0,GC=F,2333.500000,2333.500000,NaN,NaN,1,2333.500000
1,2024-05-30 05:00:00+00:00,2323.699951,2333.699951,2320.800049,2333.000000,617,GC=F,2328.354974,2328.501975,NaN,NaN,1,2328.575475
2,2024-05-30 06:00:00+00:00,2331.699951,2331.699951,2322.800049,2323.399902,1913,GC=F,2329.583230,2329.610885,NaN,NaN,1,2329.627400
3,2024-05-30 07:00:00+00:00,2355.699951,2356.899902,2331.399902,2331.399902,45741,GC=F,2337.122723,2336.529599,NaN,NaN,1,2336.243634
4,2024-05-30 08:00:00+00:00,2356.300049,2357.300049,2330.600098,2355.800049,13770,GC=F,2341.761555,2340.806212,NaN,NaN,1,2340.335542


# 4. Vérification des colonnes

Vérification de la présence des variables nécessaires à l'entraînement du modèle.

In [13]:
print(gold.columns.tolist())


['Datetime', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker', 'EMA20', 'EMA50', 'RSI', 'ATR', 'Target', 'EMA200']


# 5. Vérification des valeurs manquantes

Les indicateurs comme RSI et ATR génèrent naturellement des valeurs manquantes au début de la série temporelle.

In [11]:
gold[["EMA20","EMA50","EMA200","RSI","ATR"]].isna().sum()

EMA20      0
EMA50      0
EMA200     0
RSI       13
ATR       13
dtype: int64

# 6. Sélection des variables explicatives et de la cible

Les variables explicatives sont les indicateurs techniques.
La cible représente la direction future du marché.

In [15]:
features = [
    "EMA20",
    "EMA50",
    "EMA200",
    "RSI",
    "ATR"
]

X = gold[features]
y = gold["Target"]

print("X :", X.shape)
print("y :", y.shape)

X : (11423, 5)
y : (11423,)


# 7. Nettoyage final du jeu d'entraînement

Suppression des lignes contenant des valeurs manquantes avant l'entraînement du modèle.

In [16]:
dataset = pd.concat([X, y], axis=1)
dataset = dataset.dropna()

X = dataset[features]
y = dataset["Target"]

print("Dataset final :", dataset.shape)

Dataset final : (11410, 6)


# 8. Vérification de la distribution de la cible

Cette étape permet de vérifier si les classes Achat/Vente sont équilibrées.

In [18]:
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

Target
1    6137
0    5273
Name: count, dtype: int64
Target
1    53.786152
0    46.213848
Name: proportion, dtype: float64


# 9. Séparation des données Train/Test

Le jeu de données est séparé en données d'entraînement et données de test.

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train :", X_train.shape)
print("Test :", X_test.shape)

Train : (9128, 5)
Test : (2282, 5)


# 10. Entraînement du modèle Random Forest

Random Forest est utilisé comme premier modèle MVP pour le prototype.

In [22]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Modèle entraîné avec succès")

Modèle entraîné avec succès


# 11. Prédictions

Le modèle est utilisé pour prédire les signaux sur les données de test.

In [24]:
predictions = model.predict(X_test)

print(predictions[:20])

[0 0 1 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 0 1]


# 12. Évaluation du modèle

Évaluation du modèle à l'aide de métriques de classification.

In [27]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy :", accuracy)
print()
print(classification_report(y_test, predictions))

Accuracy : 0.7655565293602103

              precision    recall  f1-score   support

           0       0.75      0.74      0.74      1044
           1       0.78      0.79      0.79      1238

    accuracy                           0.77      2282
   macro avg       0.76      0.76      0.76      2282
weighted avg       0.77      0.77      0.77      2282



# 13. Matrice de confusion

La matrice de confusion permet de visualiser les bonnes et mauvaises prédictions.

In [28]:
cm = confusion_matrix(y_test, predictions)

print("Matrice de confusion :")
print(cm)

Matrice de confusion :
[[770 274]
 [261 977]]


# 15. Conclusion

Un premier modèle Random Forest a été entraîné pour prédire la direction future du marché à partir des indicateurs techniques.

Les résultats obtenus serviront de base pour les étapes suivantes :

- amélioration des variables ;
- comparaison avec d'autres modèles ;
- validation par backtesting ;
- intégration dans une application ou un dashboard.